In [3]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Tuple

import os
import sys
from pathlib import Path
import pandas as pd

ROOT = Path(os.path.abspath('')).resolve().parents[2]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import nvitk as nv
from nvitk.core import setup

from nvitk.transform import resample_mask_to_pet
from nvitk.segmentation.labels import get_label 

from nvitk.core import as_backend_array, to_numpy

setup(globals())

# splprep, splev = scipy.interpolate.splprep, scipy.interpolate.splev
from scipy.interpolate import splprep, splev
from skimage.graph import route_through_array

from nvitk.core.logger import Logger
log = Logger()

Adding /home/imarcoss/nvitk/src/nvitk to sys.path


In [4]:
# raw_ct = nv.imread('/data3/BIOIT_IMAGE/PESA_Fat/DATA/Visit-5-DIXON_PET-CT/DATA/NIFTI/202602_Week1/PESA11471769/CT.nii')
raw_pt = nv.imread('/data3/BIOIT_IMAGE/PESA_Fat/DATA/Visit-5-DIXON_PET-CT/DATA/NIFTI/202602_Week1/PESA11471769/PT.nii')
organs = nv.imread('/data3/BIOIT_IMAGE/PESA_Fat/DATA/Visit-5-DIXON_PET-CT/RESULTS/202602_Week1/res_segmentation_ct/PESA11471769/CT/total.nii')

# print(raw_ct)
print(raw_pt)
print(organs)

# resampled_ct = resample_mask_to_pet(raw_ct, raw_pt, order=1)
resampled_organs = resample_mask_to_pet(organs, raw_pt, order=0)

# print(resampled_ct)
print(resampled_organs)

kidney_r = get_label(resampled_organs, 2)
kidney_l = get_label(resampled_organs, 3)
bladder  = get_label(resampled_organs, 21)

Image(shape=(288, 288, 490), dtype=float64, backend=cupy, axes='XYZ', orientation='RAS', name='PT', modality='PT', submodality=None, rescale_type='DV')
Image(shape=(512, 512, 489), dtype=uint8, backend=cupy, axes='XYZ', orientation='RAS', name='total', modality='CT', submodality=None, rescale_type='DV')
Image(shape=(288, 288, 490), dtype=uint8, backend=cupy, axes='XYZ', orientation='RAS', name='total', modality='CT', submodality=None, rescale_type='DV')


In [5]:
# ---------------------------------------------------------------------------
# Anchors (heuristic)
# ---------------------------------------------------------------------------


def _convex_hull_slicewise_z(vol_uint8: np.ndarray) -> np.ndarray:
    """Same convention as stage2: hull each slice along the last axis."""
    try:
        from skimage.morphology import convex_hull_image
    except Exception:
        return vol_uint8.copy()
    vol_uint8 = to_numpy(vol_uint8)
    out = vol_uint8.astype(np.uint8, copy=True)
    for i in range(out.shape[-1]):
        sl = out[..., i]
        if sl.any():
            out[..., i] = convex_hull_image(sl)
    return as_backend_array(out)

def anchor_kidney_pelvis_concavity(
    kidney_mask: np.ndarray,
    *,
    spacing: tuple[float, float, float] = (1.0, 1.0, 1.0),
    midline_x_index: float | None = None,
    medial_half_width_vox: float | None = None,
    structure: np.ndarray | None = None,
) -> tuple[float, float, float]:
    m = as_backend_array(kidney_mask).astype(bool)
    if not m.any():
        raise ValueError("empty kidney_mask")
    hull = _convex_hull_slicewise_z(m.astype(np.uint8)) > 0
    cavity = hull & ~m
    if midline_x_index is not None and medial_half_width_vox is not None:
        xs = np.arange(m.shape[0], dtype=np.float32)[:, None, None]
        cavity &= np.abs(xs - midline_x_index) <= medial_half_width_vox
    if structure is None:
        structure = np.ones((3, 3, 3), dtype=bool)
    lab, n = ndi.label(cavity, structure=structure)
    if n < 1:
        raise ValueError("no cavity voxels after hull − mask")
    sizes = np.bincount(lab.ravel())
    sizes[0] = 0
    lid = int(sizes.argmax())
    coords = np.argwhere(lab == lid).astype(np.float64)
    return as_backend_array(coords.mean(axis=0))

def anchor_bladder_entry_per_side(
    bladder_mask: np.ndarray,
    kidney_mask: np.ndarray,
    *,
    axis_x: int = 0,
    axis_z: int = -1,
    superior_fraction: float = 0.35,
    structure: np.ndarray | None = None,
) -> tuple[float, float, float]:
    m  = as_backend_array(bladder_mask).astype(bool)
    km = as_backend_array(kidney_mask).astype(bool)
    if not m.any():
        raise ValueError("empty bladder_mask")
    if not km.any():
        raise ValueError("empty kidney_mask — cannot determine ipsilateral side")

    if axis_x < 0:
        axis_x = m.ndim + axis_x
    if axis_z < 0:
        axis_z = m.ndim + axis_z

    # 1. Lateral centroid of kidney → which side of the volume it occupies
    k_coords  = np.argwhere(km)
    k_x_mean  = float(k_coords[:, axis_x].mean())
    vol_x_mid = m.shape[axis_x] / 2.0

    # 2. Build an ipsilateral lateral mask for the bladder
    x_arr     = np.arange(m.shape[axis_x], dtype=np.float64)
    x_in_side = (x_arr >= vol_x_mid) if (k_x_mean >= vol_x_mid) else (x_arr < vol_x_mid)

    lat_mask = np.zeros_like(m, dtype=bool)
    lat_sl   = [slice(None)] * m.ndim
    lat_sl[axis_x] = x_in_side
    lat_mask[tuple(lat_sl)] = True

    cand = m & lat_mask
    if not cand.any():
        log.warning(
            "anchor_bladder_entry_per_side: no bladder voxels on the ipsilateral side "
            "(axis_x=%d, k_x=%.1f, mid=%.1f) — falling back to full bladder mask.",
            axis_x, k_x_mean, vol_x_mid,
        )
        cand = m.copy()

    # 3. Top ``superior_fraction`` in IS direction (trigone corners are superior)
    other_axes = tuple(i for i in range(m.ndim) if i != axis_z)
    z_present  = np.where(cand.any(axis=other_axes))[0]
    z_min_c, z_max_c = int(z_present.min()), int(z_present.max())
    span  = z_max_c - z_min_c + 1
    trim  = max(1, int(span * (1.0 - superior_fraction)))
    z_sup = z_max_c - trim  # lower bound of the superior band

    sup_sl = [slice(None)] * m.ndim
    sup_sl[axis_z] = slice(z_sup, z_max_c + 1)
    sup_mask = np.zeros_like(cand, dtype=bool)
    sup_mask[tuple(sup_sl)] = True

    region = cand & sup_mask
    if not region.any():
        log.warning(
            "anchor_bladder_entry_per_side: superior band empty — using full ipsilateral cand."
        )
        region = cand

    # 4. Largest CC centroid
    if structure is None:
        structure = np.ones((3, 3, 3), dtype=bool)
    lab, n = ndi.label(region, structure=structure)
    if n < 1:
        raise ValueError("no bladder voxels found in ipsilateral superior region")
    sizes    = np.bincount(lab.ravel())
    sizes[0] = 0
    lid      = int(sizes.argmax())
    coords   = np.argwhere(lab == lid).astype(np.float64)
    return coords.mean(axis=0)

In [12]:
# ---------------------------------------------------------------------------
# Cost map
# ---------------------------------------------------------------------------


def build_cost_volume(
    suv: Any,
    p_start: Any,
    p_end: Any,
    spacing_xyz_mm: Tuple[float, float, float],
    *,
    # ---- PET / gap-filling ----------------------------------------
    w_pet: float = 1.0,
    suv_clip: float = 50.0,
    suv_fill_sigma_vox: float = 4.0,
    suv_fill_blend: float = 0.5,
    eps: float = 1e-3,
    # ---- Lateral hemisphere soft guardrail ------------------------
    w_lateral: float = 1.5,
    lateral_free_mm: float = 25.0,
    lateral_slope_mm: float = 40.0,
    axis_x: int = 0,
    # ---- Distance-to-endpoint pull --------------------------------
    w_dist: float = 0.003,
    meta = None
) -> Any:
    """
    PET-primary cost volume for ureter MCP routing — v2.1.

    What changed from v2
    --------------------
    The corridor prior (distance to the straight kidney→bladder segment) has
    been **removed**.  It was structurally wrong: because it is zero on the
    straight line and grows everywhere else, the MCP minimised it by routing
    straight, producing a linear cylindrical mask regardless of the PET signal.
    A path prior that rewards the straight line *is* the straight line.

    Components
    ----------
    pet_term
        Inverse of a gap-filled SUV envelope — the primary and dominant cost.
        High uptake = low cost = preferred.  Dark spots are healed by blending
        the raw map with a smoothed version (element-wise maximum), so short
        low-uptake gaps along the ureter do not reroute the path.

    lateral_term
        A piecewise-linear (hinge) penalty in the lateral direction only.
        Free zone: ``lateral_free_mm`` (default 25 mm) around the kidney
        pelvis x-position — zero cost.  Ramp zone: cost rises at
        ``w_lateral / lateral_slope_mm`` per mm over the next
        ``lateral_slope_mm`` (default 40 mm).  Hard cap: beyond
        ``lateral_free_mm + lateral_slope_mm`` the penalty saturates at
        ``w_lateral``.

        This acts as a guardrail that blocks grossly wrong contralateral
        routes without pulling the path straight.  It has no opinion on
        IS or AP movement — the ureter is free to curve in those directions.

    dist_term
        Minimal endpoint pull — genuinely just a tiebreaker (w=0.003).
        Prevents the path from idling near a local high-SUV region by adding
        a tiny bias toward the destination that can never override the PET
        signal.

    Parameters
    ----------
    w_lateral : float
        Maximum penalty added by the lateral guardrail (plateau value).
        In the same units as pet_term.  Default 1.5 ≈ cost of a near-zero
        SUV voxel — enough to block, not enough to reshape the path.
    lateral_free_mm : float
        Half-width of the zero-penalty lateral band around the kidney x.
    lateral_slope_mm : float
        Width of the linear ramp from free band to plateau.  The full
        guardrail spans ``lateral_free_mm + lateral_slope_mm`` in mm.
    axis_x : int
        Array axis for the Lateral (L-R) direction (default 0).
    """
    s0, s1, s2 = spacing_xyz_mm

    # ---- gap-filled SUV ---------------------------------------------------
    suv_c      = np.clip(suv, suv_clip, None)
    suv_n      = (suv_c - suv_c.min()) / (suv_c.max() - suv_c.min())
    suv_smooth = ndi.gaussian_filter(suv_c.astype(np.float64), sigma=float(suv_fill_sigma_vox), mode="nearest")

    nv.imsave('pet_gaus.nii', suv_smooth, metadata=meta)
    
    # Element-wise max: hotspots preserved; dark gaps lifted by blended envelope
    suv_filled = np.maximum(suv_c, float(suv_fill_blend) * suv_smooth)
    pet_term   = float(w_pet) / (suv_filled + float(eps))

    nv.imsave('pet_term.nii', pet_term, metadata=meta)
    nv.imsave('pet_term_log.nii', np.log(pet_term), metadata=meta)

    # ---- physical coordinate grid (per-axis spacing, no physical label assumed)
    a0, a1, a2 = np.indices(suv.shape)
    s_arr = np.array([s0, s1, s2], dtype=np.float64)

    # ---- lateral guardrail ------------------------------------------------
    # Hinge penalty: zero within lateral_free_mm, linear ramp to plateau w_lateral.
    # Operates only on axis_x; has no opinion on the other two axes.
    axes       = [a0, a1, a2]
    sp_x       = float(s_arr[axis_x])
    x_start_mm = float(p_start[axis_x]) * sp_x
    x_mm       = axes[axis_x].astype(np.float64) * sp_x
    lat_dev_mm = np.abs(x_mm - x_start_mm)

    free  = float(lateral_free_mm)
    slope = float(lateral_slope_mm)
    # Piecewise: 0 if dev<=free, linear ramp if free<dev<=free+slope, plateau beyond
    ramp         = np.clip((lat_dev_mm - free) / slope, 0.0, 1.0)
    lateral_term = float(w_lateral) * ramp

    nv.imsave('lateral_term.nii', lateral_term, metadata=meta)

    # ---- minimal endpoint pull --------------------------------------------
    pe     = np.array(p_end, dtype=np.float64) * s_arr
    phys   = np.stack([
        a0.astype(np.float64) * s0,
        a1.astype(np.float64) * s1,
        a2.astype(np.float64) * s2,
    ], axis=-1)
    dist_mm   = np.sqrt(np.sum((phys - pe) ** 2, axis=-1))
    dist_term = float(w_dist) * dist_mm

    nv.imsave('dist_ter.nii', dist_term, metadata=meta)

    cost = pet_term.astype(np.float64) + lateral_term + dist_term
    # return np.clip(cost, 1e-6, None)
    return cost

In [13]:
# ---------------------------------------------------------------------------
# Path + spline + tube
# ---------------------------------------------------------------------------


def minimum_cost_path_zyx(
    cost: Any,
    start_zyx: Tuple[int, int, int],
    end_zyx: Tuple[int, int, int],
) -> Any:
    c_np   = to_numpy(cost, copy=False)
    path, _ = route_through_array(
        c_np, start_zyx, end_zyx, fully_connected=True, geometric=True
    )
    return as_backend_array(path).astype(np.int32)


def spline_resample_zyx(
    path_zyx: Any,
    n_points: int,
    s_smooth: float,
    bounds_lo: Any,
    bounds_hi: Any,
) -> Any:
    p = to_numpy(path_zyx, copy=True)
    if p.shape[0] < 4:
        rep = 4 - p.shape[0]
        p   = np.vstack([p, np.repeat(p[-1:], rep, axis=0)])
    pts     = p.T
    tck, _u = splprep(pts, s=float(s_smooth), k=3)
    u_new   = np.linspace(0, 1, int(n_points), dtype=np.float64)
    z, y, x = splev(u_new, tck)
    out     = np.stack([z, y, x], axis=1)
    out     = np.clip(out, bounds_lo, bounds_hi)
    return as_backend_array(out)


def line_mask_from_path(shape: Tuple[int, ...], path_zyx_float: Any) -> Any:
    mask = np.zeros(shape, dtype=np.uint8)
    pi   = np.round(as_backend_array(path_zyx_float)).astype(np.int32)
    for i in range(pi.shape[0]):
        z = int(np.clip(pi[i, 0], 0, shape[0] - 1))
        y = int(np.clip(pi[i, 1], 0, shape[1] - 1))
        x = int(np.clip(pi[i, 2], 0, shape[2] - 1))
        mask[z, y, x] = 1
    return mask


def edt_tube_mm(
    line_mask: Any,
    spacing_xyz_mm: Tuple[float, float, float],
    radius_mm: float,
) -> Any:
    inv     = as_backend_array(line_mask == 0)
    dist_mm = ndi.distance_transform_edt(inv, sampling=spacing_xyz_mm)
    return (dist_mm <= radius_mm).astype(np.uint8)

In [14]:
# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


def run_ureter_segmentation(
    pet_suv: "Image",
    kidney_r: "Image",
    kidney_l: "Image",
    bladder: "Image",
    *,
    # ---- Tube geometry -------------------------------------------------
    radius_mm: float = 6.0,
    # ---- Cost weights --------------------------------------------------
    w_pet: float = 5.0,
    w_dist: float = 0.0,
    w_lateral: float = 0.0,
    lateral_free_mm: float = 25.0,
    lateral_slope_mm: float = 7.0,
    # ---- Gap-filling ---------------------------------------------------
    suv_fill_sigma_vox: float = 0.5,
    suv_fill_blend: float = 0.2,
    # ---- Spline --------------------------------------------------------
    spline_s: float = 1.5,
    # ---- Axis conventions — adjust if your resampled grid differs -------
    axis_x: int = 0,   # lateral (L-R) axis in the array
    axis_z: int = -1,  # IS axis in the array
) -> Tuple[Any, Dict[str, Any], Dict[str, Any]]:
    suv = pet_suv.data
    bm  = bladder.data

    sp = pet_suv.spacing
    if sp is None or len(sp) < 3:
        raise ValueError("CT (or PET) Image must carry spacing in metadata")
    spacing_xyz = (float(sp[0]), float(sp[1]), float(sp[2]))

    ureter_data = np.zeros_like(suv, dtype=np.uint8)
    paths: Dict[str, Any]    = {}
    paths_sp: Dict[str, Any] = {}

    for side, km in [("R", kidney_r.data), ("L", kidney_l.data)]:

        # ---- anchors -------------------------------------------------------
        p_start = anchor_kidney_pelvis_concavity(km, spacing=spacing_xyz)
        p_end   = anchor_bladder_entry_per_side(
            bm, km,
            axis_x=axis_x,
            axis_z=axis_z,
        )

        start = tuple(int(round(float(v))) for v in as_backend_array(p_start))
        end   = tuple(int(round(float(v))) for v in as_backend_array(p_end))

        log.info("Ureter-%s: start_vox=%s  end_vox=%s", side, start, end)

        # ---- PET-primary cost volume ---------------------------------------
        cost = build_cost_volume(
            suv, p_start, p_end, spacing_xyz,
            w_pet=w_pet,
            w_dist=w_dist,
            w_lateral=w_lateral,
            lateral_free_mm=lateral_free_mm,
            lateral_slope_mm=lateral_slope_mm,
            suv_fill_sigma_vox=suv_fill_sigma_vox,
            suv_fill_blend=suv_fill_blend,
            axis_x=axis_x,
            meta=pet_suv.metadata
        )
        nv.imsave(f'cost_{side}.nii', cost, metadata=pet_suv.metadata)

        # ---- MCP routing ---------------------------------------------------
        path = minimum_cost_path_zyx(cost, start, end)
        log.info("Ureter-%s: MCP path length = %d voxels", side, int(path.shape[0]))

        # ---- B-spline smoothing --------------------------------------------
        n_pts   = max(2 * int(path.shape[0]), 64)
        lo      = np.array([0, 0, 0], dtype=np.float64)
        hi      = np.array(
            [suv.shape[0] - 1, suv.shape[1] - 1, suv.shape[2] - 1],
            dtype=np.float64,
        )
        path_sp = spline_resample_zyx(path, n_pts, spline_s, lo, hi)

        # ---- EDT tube mask -------------------------------------------------
        line         = line_mask_from_path(suv.shape, path_sp)
        tube         = edt_tube_mm(line, spacing_xyz, radius_mm)
        ureter_data  = np.where(tube > 0, tube, ureter_data)

        paths[side]    = path
        paths_sp[side] = path_sp

    mask_ureter = pet_suv.with_data(ureter_data.astype(np.uint8))
    return mask_ureter, paths, paths_sp

In [16]:
from nvitk.core import using

with using('cpu'):
    raw_pt = raw_pt.to_backend('cpu')
    kidney_r = kidney_r.to_backend('cpu')
    kidney_l = kidney_l.to_backend('cpu')
    bladder = bladder.to_backend('cpu')

    mask, p, p_sp = run_ureter_segmentation(raw_pt, kidney_r, kidney_l, bladder)
    print(mask)
    # print(p)
    # print(p_sp)

00:09:34 | INFO     | Ureter-R: start_vox=(172, 113, 212)  end_vox=(151, 127, 92)
00:10:08 | INFO     | Ureter-R: MCP path length = 121 voxels
00:10:18 | INFO     | Ureter-L: start_vox=(117, 106, 223)  end_vox=(137, 126, 93)
00:10:52 | INFO     | Ureter-L: MCP path length = 141 voxels


Image(shape=(288, 288, 490), dtype=uint8, backend=numpy, axes='XYZ', orientation='RAS', name='PT', modality='PT', submodality=None, rescale_type='DV')


In [17]:
nv.imsave('ureter_mask.nii', mask)